# D530 GCP Introduction

**Focus:** Google Cloud services used in data engineering.

This introduction assumes basic AWS knowledge and maps familiar AWS services to GCP. Start with where data lands, how it is processed, and where it is queried.

**Example throughout:** collect retail orders, clean the records, and calculate daily sales.

## 1. Map AWS to GCP

These are mappings by purpose; features and operating models differ.

| Data engineering need | AWS reference | GCP service |
|---|---|---|
| Store files in a data lake | Amazon S3 | Cloud Storage (GCS) |
| Run Spark / Hadoop jobs | Amazon EMR | Dataproc / Managed Service for Apache Spark |
| Managed batch and stream processing | AWS Glue ETL / Amazon Managed Service for Apache Flink, depending on workload | Dataflow (Apache Beam) |
| SQL analytics and warehousing | Amazon Redshift; Athena for querying lake data | BigQuery |
| Managed relational application database | Amazon RDS | Cloud SQL |
| PostgreSQL-compatible database platform | Amazon Aurora PostgreSQL | AlloyDB for PostgreSQL |

Dataflow is not a Spark service. BigQuery covers use cases associated with both Redshift and Athena, but is not an exact equivalent of either.

[Reference: Google Cloud service comparison](https://docs.cloud.google.com/docs/get-started/aws-azure-gcp-service-comparison)

## 2. A few GCP terms

- **Project:** groups resources and provides a boundary for permissions, APIs, and billing attribution. Example project ID: `retail-de-demo`.
- **Region:** the location selected for a resource. Choose compatible locations for storage, processing, and analytics.
- **IAM:** controls who can perform which actions on resources.
- **Service account:** an identity used by a pipeline or application to access services.

Example: a processing job uses a service account with permission to read a GCS bucket and write to a BigQuery dataset.

[Reference: Google Cloud overview](https://docs.cloud.google.com/docs/overview)

## 3. Cloud Storage (GCS) ? map from S3

Store CSV, JSON, Parquet, logs, and other files as objects in buckets. Use GCS as the landing area for incoming data and as storage for processed files.

```text
AWS:  s3://retail-lake/raw/orders/orders.csv
GCP:  gs://retail-lake-demo/raw/orders/orders.csv
```

A practical layout:

```text
gs://retail-lake-demo/
    raw/orders/         <- source files
    curated/orders/     <- cleaned Parquet files
    rejected/orders/    <- invalid records
```

**Use it when:** files arrive from applications, exports, or external systems. A processing or query service reads those files; the bucket itself does not run transformations.

[Reference: Cloud Storage overview](https://docs.cloud.google.com/storage/docs/introduction)

## 4. Dataproc ? map from EMR

Run Spark and Hadoop workloads on managed infrastructure. Current documentation uses **Managed Service for Apache Spark**; Dataproc is the familiar name used in this introduction.

**Example:** read order files from GCS, remove duplicates with PySpark, join product data, and write clean Parquet back to GCS.

```text
GCS raw files -> Spark job on Dataproc -> GCS curated files
```

**Use it when:** an existing pipeline uses Spark, PySpark, or Hadoop, or transformations need Spark libraries.

Cluster-based execution provides control over the environment. A serverless Spark option runs supported workloads without maintaining a cluster.

[Reference: Managed Spark overview](https://docs.cloud.google.com/managed-spark/docs/concepts/clusters-overview)

## 5. Dataflow ? managed batch and streaming pipelines

Dataflow executes **Apache Beam** pipelines. Use it to read records, validate them, transform them, and write results to a destination.

```text
Batch:      GCS files -> Dataflow -> BigQuery
Streaming:  Pub/Sub events -> Dataflow -> BigQuery
```

**Example:** process order events as they arrive, route invalid events to a separate destination, and calculate sales in five-minute windows.

Pub/Sub carries messages from producers to consumers; Dataflow performs the processing.

**AWS mapping:** compare the workload with Glue ETL for batch processing or Managed Service for Apache Flink for streaming. The APIs and execution engines differ.

**Choose by code:** PySpark points toward Dataproc; Apache Beam points toward Dataflow.

[Reference: Dataflow](https://cloud.google.com/products/dataflow)

## 6. BigQuery ? SQL analytics

BigQuery is a managed, serverless analytics warehouse. Load data into tables and use SQL to filter, join, and aggregate it. External tables can also query supported data stored outside BigQuery, including GCS.

Organize tables as:

```text
project.dataset.table
retail-de-demo.analytics.orders
```

**Example:** calculate daily revenue from cleaned orders. Run this illustrative SQL in the BigQuery editor after creating and loading the table:

```sql
SELECT
    DATE(order_timestamp) AS order_date,
    SUM(order_amount) AS daily_revenue
FROM `retail-de-demo.analytics.orders`
GROUP BY order_date
ORDER BY order_date;
```

**Use it when:** the main task is analytical SQL or reporting across many records. For SQL-friendly transformations, load data and transform it directly in BigQuery.

[Reference: BigQuery overview](https://docs.cloud.google.com/bigquery/docs/introduction)

## 7. Cloud SQL and AlloyDB ? map from RDS and Aurora

| Service | Purpose | Retail example |
|---|---|---|
| Cloud SQL | Managed MySQL, PostgreSQL, or SQL Server | Store customer accounts and application orders |
| AlloyDB for PostgreSQL | Managed PostgreSQL-compatible database platform | Run a demanding PostgreSQL application workload |
| BigQuery | Analytical warehouse | Analyze historical orders and produce sales reports |

**RDS -> Cloud SQL** is the starting mapping. For **Aurora PostgreSQL**, compare AlloyDB based on application requirements; it is not a drop-in service replacement.

Cloud SQL supports application transactions such as creating an order or updating its status. An ingestion pipeline copies relevant data into BigQuery for analytical reporting.

```text
Application -> Cloud SQL -> ingestion pipeline -> BigQuery -> Reports
```

[Cloud SQL overview](https://docs.cloud.google.com/sql/docs/introduction) ? [AlloyDB overview](https://docs.cloud.google.com/alloydb/docs/overview)

## 8. Put the services together

### Daily sales from files

```text
Order exports
     |
     v
GCS: raw files
     |
     v
Dataproc: clean and join with Spark
     |
     v
GCS: curated Parquet
     |
     v
BigQuery: load tables and calculate daily sales
```

If the cleaning can be expressed easily in SQL, load the files into BigQuery and transform them there. A pipeline does not need every service.

### Sales as events arrive

```text
Order events -> Pub/Sub -> Dataflow -> BigQuery -> Dashboard
```

Use Dataflow when events need transformations or windowed calculations before reaching the analytical tables.

[Reference: Loading data into BigQuery](https://docs.cloud.google.com/bigquery/docs/loading-data)

## 9. Pick the service

| Requirement | Start with |
|---|---|
| Keep original CSV and Parquet files | GCS |
| Move an existing PySpark job from EMR | Dataproc |
| Transform events continuously with Apache Beam | Dataflow |
| Join tables and calculate sales with SQL | BigQuery |
| Run an application on managed PostgreSQL, MySQL, or SQL Server | Cloud SQL |
| Evaluate a PostgreSQL-compatible platform for demanding workloads | AlloyDB |

**Remember:** GCS stores files; Dataproc runs Spark; Dataflow runs Beam pipelines; BigQuery handles analytical SQL; Cloud SQL and AlloyDB serve relational application workloads.